# LLM Orchestra — MIDI + Soundfonts

Deux niveaux dans un notebook :

- **Niveau 1** : export MIDI standard (.mid) — importable dans Ableton, Logic, MuseScore, GarageBand
- **Niveau 2** : rendu WAV automatique via FluidSynth + soundfont libre

Trois genres générés : opéra (4 actes), house (128 bars), acid (pattern par layer)

| Track | Instrument GM | Source |
|-------|--------------|--------|
| 0 | Violin (40) | q_proj SVD |
| 1 | Viola (41) | k_proj SVD |
| 2 | Cello (42) | v_proj SVD |
| 3 | Contrabass (43) | down_proj row norms |
| 4 | French Horn (60) | gate_proj histos |
| 5 | Oboe (68) | up_proj histos |
| 6 | Clarinet (71) | up_proj histos |
| 7 | Harp (46) | layernorm gamma |
| 8 | Organ (19) | final_norm |
| 9 | Synth Bass (38) | down_proj (acid/house) |
| 10 | Drums (ch10) | gate stats |


In [1]:
# Option A : La plus simple et recommandée (Colab ou notebook local)
from huggingface_hub import login

# Méthode 1 : Coller ton token directement (attention : ne partage jamais le notebook !)
# Crée un token READ ici : https://huggingface.co/settings/tokens
# → New token → Read → Generate → copie-le
your_hf_token = "YOUR_HF_TOKEN_HERE"   # Remplace par TON token

login(token=your_hf_token)
print("Login Hugging Face réussi ! Tu devrais maintenant pouvoir charger les modèles gated.")

Login Hugging Face réussi ! Tu devrais maintenant pouvoir charger les modèles gated.


In [2]:
!pip install -q transformers accelerate scikit-learn scipy numpy pretty_midi
!apt-get install -q -y fluidsynth fluid-soundfont-gm
!pip install -q pyfluidsynth

import pretty_midi, os, glob
print("pretty_midi:", pretty_midi.__version__)

sf_paths = (glob.glob("/usr/share/sounds/sf2/*.sf2") +
            glob.glob("/usr/share/soundfonts/*.sf2") +
            glob.glob("/usr/share/sounds/*.sf2"))
SOUNDFONT = sf_paths[0] if sf_paths else None
print("Soundfont:", SOUNDFONT if SOUNDFONT else "non trouvee")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 52.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.9 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libdouble-conversion3 libevdev2 libfluidsynth3
  libgtk-3-0 libgtk-3-bin libgtk-3-common libgudev-1.0-0 libinput-bin
  libinput10 libinstpatch-1.0-2 libmd4c0 libmtdev1 libqt5core5a libqt5dbus5
  libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 librsvg2-common
  libwacom-bin libwacom-common libwacom9 libxcb-icccm4 libxcb-image0
  libxcb-keysyms1 libxcb-render-util0 libxcb-util1 libxcb-xinerama0
  libxcb-xinput0 libxcb-xkb1 libxcomposite1 libxkbcommon-x11-0 libxtst6 qsynth
  qt5-gtk-platformtheme qttranslations5-l10n session-migration
Suggested packages:
  

In [3]:
import numpy as np
import torch
import pretty_midi
from scipy.io import wavfile
from scipy.signal import butter, sosfilt
from scipy.stats import kurtosis as sp_kurt, skew as sp_skew
from sklearn.decomposition import TruncatedSVD, PCA
from IPython.display import Audio, display
import gc, time, warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

MODEL_ID    = "google/gemma-3-270m"
SR          = 44100
MIDI_OPERA  = "llm_opera.mid"
MIDI_HOUSE  = "llm_house.mid"
MIDI_ACID   = "llm_acid.mid"
WAV_OPERA   = "llm_opera_sf.wav"
WAV_HOUSE   = "llm_house_sf.wav"

TEMPO_OPERA = 66
TEMPO_HOUSE = 128
TEMPO_ACID  = 138

GM = {
    "violin":40,"viola":41,"cello":42,"contrabass":43,
    "strings":48,"slow_strings":49,
    "horn":60,"trumpet":56,"trombone":57,"tuba":58,"brass":61,
    "oboe":68,"clarinet":71,"bassoon":70,"flute":73,
    "harp":46,"celesta":8,"organ":19,"reed_organ":20,
    "synth_bass1":38,"synth_bass2":39,
    "lead_square":80,"lead_saw":81,
    "pad_strings":92,"pad_choir":91,
}
VEL = {"ppp":16,"pp":33,"p":49,"mp":64,"mf":80,"f":96,"ff":112,"fff":127}

def norm01(x):
    x = np.asarray(x,dtype=float)
    mn,mx = x.min(),x.max()
    return (x-mn)/(mx-mn+1e-9)

def hz_to_midi(hz):   return 69+12*np.log2(np.clip(float(hz),20,20000)/440)
def midi_to_hz(m):    return 440.0*2**((float(m)-69)/12)

MODES = {
    "major":          [0,2,4,5,7,9,11],
    "natural_minor":  [0,2,3,5,7,8,10],
    "dorian":         [0,2,3,5,7,9,10],
    "phrygian":       [0,1,3,5,7,8,10],
    "harmonic_minor": [0,2,3,5,7,8,11],
    "lydian":         [0,2,4,6,7,9,11],
    "mixolydian":     [0,2,4,5,7,9,10],
    "pentatonic_minor":[0,3,5,7,10],
}

def build_scale(root, mode, lo=24, hi=108):
    ivs = MODES[mode]
    return np.array(sorted(set(
        n for o in range(9) for st in ivs
        if lo <= (n:=root+o*12+st) <= hi
    )))

def snap(midi_f, scale):
    return int(scale[np.argmin(np.abs(scale-float(midi_f)))])

def velf(v, lo="pp", hi="ff"):
    return int(VEL[lo]+float(v)*(VEL[hi]-VEL[lo]))

print("Config OK — BPM opera=%d house=%d acid=%d" % (TEMPO_OPERA,TEMPO_HOUSE,TEMPO_ACID))


Config OK — BPM opera=66 house=128 acid=138


In [4]:
from transformers import AutoModelForCausalLM

print("Chargement", MODEL_ID)
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
model.eval()
sd  = {k:v.detach().float().numpy() for k,v in model.state_dict().items()}
NL  = model.config.num_hidden_layers
print(" %d layers — %.1fs" % (NL, time.time()-t0))
del model; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

def W(l,name):
    keys=[k for k in sd if ("layers.%d."%l) in k and name in k]
    return sd[keys[0]] if keys else None

# PCA cross-layer -> tonalite
gammas=[W(l,"input_layernorm") for l in range(NL) if W(l,"input_layernorm") is not None]
G = np.stack(gammas,0).astype(np.float32)
pca = PCA(n_components=3).fit(G)
pc_norms = np.linalg.norm(pca.components_,axis=1)
root_midi = int(np.round(36+norm01(pc_norms)[0]*24))
pc2_skew  = float(sp_skew(pca.components_[1]))
pc3_kurt  = float(sp_kurt(pca.components_[2]))
if   pc2_skew >  0.1: MODE="major"
elif pc2_skew < -0.1: MODE="natural_minor"
elif pc3_kurt >  3.0: MODE="harmonic_minor"
else:                 MODE="dorian"
SCALE = build_scale(root_midi, MODE)
names = ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]
print("Tonalite: %s %s  (MIDI %d = %.1fHz)" % (names[root_midi%12],MODE,root_midi,midi_to_hz(root_midi)))

# SVD par tenseur
SV={}
for tn in ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]:
    rows=[]
    for l in range(NL):
        Wt=W(l,tn)
        if Wt is not None and Wt.ndim==2:
            k=min(16,min(Wt.shape)-1)
            sv=TruncatedSVD(n_components=max(k,2),random_state=42).fit(Wt).singular_values_[:16] if k>=2 else np.ones(16)
            if len(sv)<16: sv=np.pad(sv,(0,16-len(sv)))
            rows.append(sv)
        else: rows.append(np.ones(16))
    SV[tn]=np.array(rows)
print("SVD extrait pour",list(SV.keys()))

# Tension dramatique (ratio attn/mlp)
tension=[]
for l in range(NL):
    an=[np.linalg.norm(W(l,t)) for t in ["q_proj","k_proj","v_proj","o_proj"] if W(l,t) is not None]
    mn=[np.linalg.norm(W(l,t)) for t in ["gate_proj","up_proj","down_proj"]   if W(l,t) is not None]
    tension.append(np.mean(an)/(np.mean(mn)+1e-9) if an and mn else 1.0)
TENSION=norm01(np.array(tension))
print("Tension peak au layer",int(np.argmax(TENSION)))

# Histogrammes (32 bins)
HISTOS={}
for tn in ["gate_proj","up_proj"]:
    rows=[]
    for l in range(NL):
        Wt=W(l,tn)
        if Wt is not None:
            flat=Wt.ravel()
            lo,hi=np.percentile(flat,2),np.percentile(flat,98)
            c,_=np.histogram(flat,bins=32,range=(lo,hi))
            rows.append(c.astype(float)/(c.sum()+1e-9))
        else: rows.append(np.ones(32)/32)
    HISTOS[tn]=np.array(rows)

# Row norms down_proj
ROW_NORMS=[]
for l in range(NL):
    Wt=W(l,"down_proj")
    if Wt is not None and Wt.ndim==2:
        step=max(1,Wt.shape[0]//16)
        rn=np.linalg.norm(Wt[::step][:16],axis=1)
        if len(rn)<16: rn=np.pad(rn,(0,16-len(rn)))
        ROW_NORMS.append(rn)
    else: ROW_NORMS.append(np.ones(16))
ROW_NORMS=np.array(ROW_NORMS)
print("Extraction complete")


Chargement google/gemma-3-270m


config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/133 [00:00<?, ?B/s]

 18 layers — 15.4s
Tonalite: C major  (MIDI 60 = 261.6Hz)
SVD extrait pour ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
Tension peak au layer 10
Extraction complete


In [5]:
def make_inst(midi_obj, program, name, is_drum=False):
    inst=pretty_midi.Instrument(program=program,is_drum=is_drum,name=name)
    midi_obj.instruments.append(inst)
    return inst

def add_notes(inst, notes_midi, t_start, dur_s, velocity=80, legato=0.95):
    gate=dur_s*legato
    for n in notes_midi:
        inst.notes.append(pretty_midi.Note(
            velocity=int(np.clip(velocity,1,127)),
            pitch=int(np.clip(n,0,127)),
            start=float(t_start),
            end=float(t_start+gate)))

def add_chord(inst, root, quality, scale, t_start, dur_s, velocity=80, octave=0, legato=0.95):
    ivs={"major":[0,4,7],"minor":[0,3,7],"dim":[0,3,6],
         "maj7":[0,4,7,11],"min7":[0,3,7,10],"dom7":[0,4,7,10],
         "sus4":[0,5,7]}.get(quality,[0,4,7])
    notes=[snap(root+i+octave*12,scale) for i in ivs]
    add_notes(inst,notes,t_start,dur_s,velocity=velocity,legato=legato)

def add_drum(midi_obj, hit, t_s, vel=100):
    dm={"kick":36,"snare":38,"clap":39,"hat_c":42,"hat_o":46,
        "crash":49,"ride":51,"rim":37,"tom_hi":50,"tom_lo":45}
    pitch=dm.get(hit,hit) if isinstance(hit,str) else hit
    drm=next((i for i in midi_obj.instruments if i.is_drum),None)
    if drm is None:
        drm=pretty_midi.Instrument(program=0,is_drum=True,name="Drums")
        midi_obj.instruments.append(drm)
    drm.notes.append(pretty_midi.Note(
        velocity=int(np.clip(vel,1,127)),pitch=int(pitch),
        start=float(t_s),end=float(t_s+0.05)))

def sv_to_notes(sv_row, scale, lo_f=0.2, hi_f=0.8, n=None):
    v=norm01(np.log1p(np.abs(sv_row)))
    if n: v=v[np.linspace(0,len(v)-1,n).astype(int)]
    lo=scale[int(lo_f*len(scale))]; hi=scale[int(hi_f*len(scale))]
    return [snap(lo+vi*(hi-lo),scale) for vi in v]

def histo_chord(h, scale, n=4):
    peaks=np.argsort(h)[-n:][::-1]
    lo=scale[len(scale)//5]; hi=scale[4*len(scale)//5]
    return sorted(set(snap(lo+(p/31)*(hi-lo),scale) for p in peaks))

print("Helpers MIDI OK")


Helpers MIDI OK


In [6]:
print("=== OPERA MIDI ===")
mop = pretty_midi.PrettyMIDI(initial_tempo=TEMPO_OPERA)
vl1=make_inst(mop,GM["violin"],"Violins I")
vl2=make_inst(mop,GM["viola"],"Violins II Viola")
vc =make_inst(mop,GM["cello"],"Cellos")
cb =make_inst(mop,GM["contrabass"],"Contrabasses")
hrn=make_inst(mop,GM["horn"],"French Horns")
ob =make_inst(mop,GM["oboe"],"Oboes")
cl =make_inst(mop,GM["clarinet"],"Clarinets")
harp=make_inst(mop,GM["harp"],"Harp")
org =make_inst(mop,GM["organ"],"Organ")

beat_o=60.0/TEMPO_OPERA; meas_o=beat_o*4

# Embed SVD pour tonalite de base
ek=[k for k in sd if "embed_tokens" in k][0]
sv_emb=TruncatedSVD(n_components=48,random_state=42).fit(sd[ek]).singular_values_
sv_emb_n=norm01(np.log1p(sv_emb))

# ACTE I : Ouverture (cordes seules)
print("Acte I...")
ACT1=96; t=0.0
for m in range(ACT1):
    prog=m/ACT1
    sv_v=sv_emb_n[int(prog*(len(sv_emb_n)-1))]
    lo=SCALE[int(len(SCALE)*0.10)]; hi=SCALE[int(len(SCALE)*0.55)]
    rn=snap(lo+sv_v*(hi-lo),SCALE)
    q="minor" if m%3!=0 else "major"
    vel=velf(prog*0.6+0.1,"ppp","mf")
    if m%2==0:
        add_chord(vl1,rn+12,q,SCALE,t,meas_o*2,velocity=vel,legato=0.97)
        add_chord(vl2,rn+7, q,SCALE,t,meas_o*2,velocity=int(vel*0.85),legato=0.97)
        add_chord(vc, rn,   q,SCALE,t,meas_o*2,velocity=int(vel*0.88),legato=0.97)
        add_chord(cb, rn-12,q,SCALE,t,meas_o*2,velocity=int(vel*0.75),legato=0.97)
        t+=meas_o*2
    if m%3==0 and prog>0.25:
        for i in range(4):
            add_notes(harp,[snap(rn+i*5,SCALE)],t-meas_o*2+i*beat_o*0.5,beat_o*0.4,
                      velocity=int(vel*0.5),legato=0.7)
    if prog>0.65:
        add_notes(ob,[snap(rn+19,SCALE)],t-meas_o*2,meas_o*2,velocity=int(vel*0.6),legato=0.93)
act1_end=t
print("  Acte I: %.1fmin" % (act1_end/60))

# ACTE II : Fuguette (canon q/k/v_proj)
print("Acte II...")
ACT2=96; CD=NL; t=act1_end

def fugue_theme(sv_mat,layer,voice,scale):
    sv=sv_mat[layer%len(sv_mat)]
    lf,hf={"soprano":(0.55,0.92),"alto":(0.40,0.78),
           "tenor":(0.25,0.63),"bass":(0.08,0.45)}.get(voice,(0.3,0.7))
    return sv_to_notes(sv,scale,lf,hf,n=4)

for m in range(ACT2):
    ct=act1_end+m*meas_o; prog=m/ACT2; l=m%NL
    ten=float(TENSION[l])
    vel=velf(prog*0.5+ten*0.3,"p","ff")
    ns1=fugue_theme(SV["q_proj"],l,"soprano",SCALE)
    for i,n in enumerate(ns1):
        add_notes(vl1,[n],ct+i*beat_o,beat_o,velocity=vel,legato=0.92)
    if m>=CD:
        ns2=fugue_theme(SV["k_proj"],l,"tenor",SCALE)
        for i,n in enumerate(ns2):
            add_notes(vc,[n],ct+i*beat_o,beat_o,velocity=int(vel*0.88),legato=0.90)
    if m>=CD*2:
        ns3=fugue_theme(SV["v_proj"],l,"alto",SCALE)
        for i,n in enumerate(ns3):
            add_notes(vl2,[n],ct+i*beat_o,beat_o,velocity=int(vel*0.82),legato=0.91)
    if m>=CD*3:
        ns4=fugue_theme(SV["o_proj"],l,"bass",SCALE)
        for i,n in enumerate(ns4):
            add_notes(cb,[n],ct+i*beat_o,beat_o,velocity=int(vel*0.78),legato=0.89)
    if ten>0.55 and m>=CD:
        hc=histo_chord(HISTOS["gate_proj"][l],SCALE,n=2)
        add_notes(hrn,hc,ct,meas_o,velocity=int(vel*0.75*ten),legato=0.88)
    if prog>0.4:
        ww=histo_chord(HISTOS["up_proj"][l],SCALE,n=1)
        if ww:
            add_notes(ob,[ww[0]+12],ct,beat_o*2,velocity=int(vel*0.60),legato=0.85)
            add_notes(cl,[ww[0]+7],ct+beat_o*2,beat_o*2,velocity=int(vel*0.55),legato=0.85)
act2_end=act1_end+ACT2*meas_o
print("  Acte II: %.1fmin" % ((act2_end-act1_end)/60))

# ACTE III : Climax (tutti layer par layer)
print("Acte III...")
MPL=6; t=act2_end
for l in range(NL):
    ten=float(TENSION[l]); prog=l/(NL-1); ct=act2_end+l*MPL*meas_o; dur=MPL*meas_o
    vm=velf(ten*0.7+0.2,"mp","fff")
    n1=snap(SCALE[int(norm01(SV["q_proj"][l])[0]*(len(SCALE)-1)*0.65+len(SCALE)*0.25)],SCALE)
    n2=snap(SCALE[int(norm01(SV["k_proj"][l])[0]*(len(SCALE)-1)*0.55+len(SCALE)*0.20)],SCALE)
    n3=snap(SCALE[int(norm01(SV["v_proj"][l])[0]*(len(SCALE)-1)*0.50+len(SCALE)*0.10)],SCALE)
    n4=snap(SCALE[int(norm01(ROW_NORMS[l])[0]*len(SCALE)*0.35)]-12,SCALE)
    add_notes(vl1,[n1],ct,dur,velocity=vm,legato=0.96)
    add_notes(vl2,[n2],ct,dur,velocity=int(vm*0.88),legato=0.96)
    add_notes(vc, [n3],ct,dur,velocity=int(vm*0.90),legato=0.96)
    add_notes(cb, [n4],ct,dur,velocity=int(vm*0.80),legato=0.96)
    if ten>0.40:
        hc=histo_chord(HISTOS["gate_proj"][l],SCALE,n=2)
        add_notes(hrn,hc,ct,dur,velocity=int(vm*0.85*ten),legato=0.90)
    if ten>0.30:
        wc=histo_chord(HISTOS["up_proj"][l],SCALE,n=2)
        add_notes(ob,[wc[0]+12] if wc else [n1+7],ct,beat_o*2,velocity=int(vm*0.70),legato=0.88)
        add_notes(cl,[wc[-1]+7] if wc else [n1+4],ct+beat_o*2,beat_o*2,velocity=int(vm*0.65),legato=0.88)
    gn=W(l,"input_layernorm")
    if gn is not None:
        for i,v in enumerate(norm01(np.abs(gn)[:4])):
            hn=snap(SCALE[int(v*(len(SCALE)-1))],SCALE)
            add_notes(harp,[hn],ct+i*beat_o*0.5,beat_o*0.4,velocity=int(vm*0.50),legato=0.70)
    if prog>0.78:
        q2="major" if MODE=="major" else "minor"
        fv=int(vm*(1.0-(prog-0.78)/0.22*0.6))
        add_chord(org,root_midi,q2,SCALE,ct,dur,velocity=fv,legato=0.98)
act3_end=act2_end+NL*MPL*meas_o
print("  Acte III: %.1fmin" % ((act3_end-act2_end)/60))

# ACTE IV : Denouement (Frobenius decay)
print("Acte IV...")
all_frob=sorted([(k,float(np.linalg.norm(v))) for k,v in sd.items() if v.ndim>=1],key=lambda x:-x[1])
frob_v=norm01(np.log1p(np.array([x[1] for x in all_frob])))
N4=min(len(frob_v),80)
for i in range(N4):
    prog=i/N4; amp=float(frob_v[i])*(1.0-prog*0.8); ct=act3_end+i*meas_o
    vd=velf(amp,"ppp","mp")
    nlo=SCALE[int(len(SCALE)*0.15)]; nhi=SCALE[int(len(SCALE)*0.65)]
    nn=snap(nlo+frob_v[i]*(nhi-nlo),SCALE)
    if prog<0.85: add_notes(vl1,[nn+12],ct,meas_o,velocity=vd,legato=0.97)
    if prog<0.70: add_notes(vl2,[nn+7], ct,meas_o,velocity=int(vd*0.85),legato=0.97)
    if prog<0.92: add_notes(vc, [nn],   ct,meas_o,velocity=int(vd*0.88),legato=0.97)
    if prog<0.97: add_notes(cb, [nn-12],ct,meas_o,velocity=int(vd*0.75),legato=0.97)
    if prog<0.40 and i%3==0:
        for j in range(4):
            add_notes(harp,[snap(nn+j*5,SCALE)],ct+j*beat_o*0.5,beat_o*0.4,
                      velocity=int(vd*0.45),legato=0.65)
    if prog>0.88:
        q3="major" if MODE=="major" else "minor"
        add_chord(org,root_midi,q3,SCALE,ct,meas_o,velocity=int(vd*0.60),legato=0.99)
total_op=act3_end+N4*meas_o
print("  Acte IV: %.1fmin" % (N4*meas_o/60))
print("  TOTAL OPERA: %.1fmin" % (total_op/60))
mop.write(MIDI_OPERA)
print("Export:", MIDI_OPERA)


=== OPERA MIDI ===
Acte I...
  Acte I: 5.8min
Acte II...
  Acte II: 5.8min
Acte III...
  Acte III: 6.5min
Acte IV...
  Acte IV: 4.8min
  TOTAL OPERA: 23.0min
Export: llm_opera.mid


In [7]:
print("=== HOUSE MIDI ===")
mhs=pretty_midi.PrettyMIDI(initial_tempo=TEMPO_HOUSE)
bass_h=make_inst(mhs,GM["synth_bass1"],"Bass")
pad_h =make_inst(mhs,GM["pad_strings"],"Pad")
lead_h=make_inst(mhs,GM["lead_saw"],"Lead")
beat_h=60.0/TEMPO_HOUSE; step_h=beat_h/4; bar_h=beat_h*4

SH=build_scale(root_midi,"natural_minor",lo=28,hi=84)
SB=build_scale(root_midi-12,"pentatonic_minor",lo=24,hi=60)

# Progressions depuis q_proj SVD
hchords=[snap(SH[int(len(SH)*0.3)]+norm01(SV["q_proj"][i*(NL//8)%NL])[0]*(SH[int(len(SH)*0.7)]-SH[int(len(SH)*0.3)]),SH) for i in range(8)]

NBARS=128
for bar in range(NBARS):
    tb=bar*bar_h; prog=bar/NBARS
    sec="intro" if bar<16 else "buildup" if bar<32 else "drop" if bar<96 else "outro"
    cn=hchords[(bar//8)%len(hchords)]
    if bar%2==0:
        pv={"intro":35,"buildup":50,"drop":70,"outro":45}[sec]
        pns=[snap(cn+i,SH) for i in [0,3,7,12]]
        for n in pns:
            pad_h.notes.append(pretty_midi.Note(velocity=pv,pitch=int(np.clip(n,0,127)),start=tb,end=tb+bar_h*2*0.98))
    if sec in ["buildup","drop","outro"] and not(sec=="buildup" and bar<24):
        rnn=norm01(ROW_NORMS[bar%NL])
        bv={"buildup":60,"drop":90,"outro":70}[sec]
        for s in range(16):
            fn=snap(SB[int(rnn[s%16]*len(SB)*0.85)],SB)
            if s in [3,7,11,15]: fn=min(fn+12,72)
            bass_h.notes.append(pretty_midi.Note(velocity=int(np.clip(bv+(20 if s in[0,8] else 0),1,127)),
                pitch=int(np.clip(fn,0,127)),start=tb+s*step_h,end=tb+s*step_h+step_h*0.85))
    if sec=="drop" or (sec=="buildup" and bar>=28):
        for i in range(16):
            an=snap(cn+[12,15,19,24][i%4],SH)
            lead_h.notes.append(pretty_midi.Note(velocity=65,pitch=int(np.clip(an,0,127)),
                start=tb+i*step_h,end=tb+i*step_h+step_h*0.88))
    if sec in ["buildup","drop","outro"]:
        hv=int(50+TENSION[bar%NL]*60)
        for b in range(4):
            ts=tb+b*beat_h
            if b in[0,2]: add_drum(mhs,"kick",ts,110)
            if b in[1,3] and (sec=="drop" or (sec=="buildup" and bar>=24) or sec=="outro"): add_drum(mhs,"snare",ts,95)
        for e in range(8): add_drum(mhs,"hat_o" if e in[2,6] else "hat_c",tb+e*beat_h/2,hv)
mhs.write(MIDI_HOUSE)
print("Export: %s (%.1fmin)" % (MIDI_HOUSE, NBARS*bar_h/60))

print("=== ACID MIDI ===")
mac=pretty_midi.PrettyMIDI(initial_tempo=TEMPO_ACID)
acid_b=make_inst(mac,GM["synth_bass2"],"Acid Bass")
beat_a=60.0/TEMPO_ACID; step_a=beat_a/4; bar_a=beat_a*4
SA=build_scale(root_midi,"phrygian",lo=28,hi=76)

for layer in range(NL):
    prog=layer/NL; n_reps=2+int(prog*4)
    rnn=norm01(ROW_NORMS[layer])
    t_layer=layer*bar_a*6
    for rep in range(n_reps):
        tr=t_layer+rep*bar_a
        for s in range(16):
            ts=tr+s*step_a
            nn=snap(SA[int(rnn[s%16]*(len(SA)-1))],SA)
            if s in[3,7]: nn=min(nn+12,76)
            acc=rnn[s%16]>0.55 or s in[0,4,8,12]
            vv=int(np.clip((90 if acc else 65)*(0.4+prog*0.6),1,127))
            acid_b.notes.append(pretty_midi.Note(velocity=vv,pitch=int(np.clip(nn,0,127)),
                start=ts,end=ts+step_a*(0.95 if acc else 0.85)))
        hv2=int(50+TENSION[layer]*60)
        for b in range(4):
            add_drum(mac,"kick",tr+b*beat_a,115)
            add_drum(mac,"snare",tr+(b+0.5)*beat_a,100)
        for e in range(8): add_drum(mac,"hat_o" if e in[2,6] else "hat_c",tr+e*beat_a/2,hv2)
mac.write(MIDI_ACID)
print("Export: %s (%.1fmin)" % (MIDI_ACID, NL*6*bar_a/60))


=== HOUSE MIDI ===
Export: llm_house.mid (4.0min)
=== ACID MIDI ===
Export: llm_acid.mid (3.1min)


In [8]:
import subprocess, os

# Verifier / telecharger soundfont
if SOUNDFONT is None or not os.path.exists(SOUNDFONT):
    print("Telechargement GeneralUser GS SF2...")
    url="https://github.com/cjthompson/generaluser/releases/latest/download/GeneralUser_GS_v1.471.sf2"
    subprocess.run(["wget","-q","-O","/tmp/GeneralUser.sf2",url])
    SOUNDFONT="/tmp/GeneralUser.sf2" if os.path.exists("/tmp/GeneralUser.sf2") else SOUNDFONT

def render_midi(midi_path, wav_path, sf=SOUNDFONT, sr=44100):
    if not sf or not os.path.exists(sf):
        print("  Soundfont introuvable, rendu skip"); return False
    cmd=["fluidsynth","-ni","-g","1.0","-r",str(sr),"-F",wav_path,sf,midi_path]
    print("  Rendu",midi_path,"...")
    r=subprocess.run(cmd,capture_output=True,text=True)
    if r.returncode!=0: print("  Erreur:",r.stderr[:200]); return False
    mb=os.path.getsize(wav_path)/1e6
    print("  OK:",wav_path,"(%.0fMB)" % mb); return True

render_midi(MIDI_OPERA, WAV_OPERA)
render_midi(MIDI_HOUSE, WAV_HOUSE)


  Rendu llm_opera.mid ...
  OK: llm_opera_sf.wav (244MB)
  Rendu llm_house.mid ...
  OK: llm_house_sf.wav (42MB)


True

In [9]:
import os
from scipy.io import wavfile
from scipy.signal import butter, sosfilt

def master(path_in, path_out, style="orchestra"):
    if not os.path.exists(path_in): print("Fichier absent:",path_in); return
    sr,data=wavfile.read(path_in)
    audio=data.astype(np.float64)/(32768.0 if data.dtype==np.int16 else 1.0)
    if audio.ndim==1: audio=np.stack([audio,audio],axis=1)
    for ch in range(2):
        sos=butter(1,28,btype="high",fs=sr,output="sos")
        audio[:,ch]=sosfilt(sos,audio[:,ch])
        if style=="orchestra":
            sos_m=butter(1,[2800,5500],btype="band",fs=sr,output="sos")
            audio[:,ch]+=sosfilt(sos_m,audio[:,ch])*0.10
            sos_h=butter(2,16000,btype="low",fs=sr,output="sos")
            audio[:,ch]=sosfilt(sos_h,audio[:,ch])
        elif style=="house":
            sos_b=butter(1,[60,200],btype="band",fs=sr,output="sos")
            audio[:,ch]+=sosfilt(sos_b,audio[:,ch])*0.15
    audio=np.tanh(audio*1.4)/np.tanh(1.4)
    peak=np.abs(audio).max()
    audio=audio/(peak+1e-9)*0.891
    out=(audio*31500).astype(np.int16)
    wavfile.write(path_out,sr,out)
    print("Master OK:",path_out,"(%.0fMB, %.1fmin)" % (out.nbytes/1e6,len(out)/sr/60))

master(WAV_OPERA,"llm_opera_master.wav","orchestra")
master(WAV_HOUSE,"llm_house_master.wav","house")


Master OK: llm_opera_master.wav (244MB, 23.0min)
Master OK: llm_house_master.wav (42MB, 4.0min)


In [ ]:
from IPython.display import Audio, display
from google.colab import files
from scipy.io import wavfile
import os

def preview(path, label, start=0, dur=90):
    if not os.path.exists(path): print("Non trouve:",path); return
    sr,data=wavfile.read(path)
    s=int(start*sr); e=min(int((start+dur)*sr),len(data))
    chunk=data[s:e]
    print(label+"  (apercu %ds depuis %.0fs)" % (dur,start))
    display(Audio(chunk.T if chunk.ndim==2 else chunk,rate=sr))

beat_o=60.0/TEMPO_OPERA; meas_o=beat_o*4
preview("llm_opera_master.wav","Opera - Acte I",start=0)
preview("llm_opera_master.wav","Opera - Acte II",start=96*meas_o)
preview("llm_house_master.wav","House - Drop",start=32*60.0/TEMPO_HOUSE*4)

print("Telechargement MIDI:")
for f in [MIDI_OPERA,MIDI_HOUSE,MIDI_ACID]:
    if os.path.exists(f): print(" ",f); files.download(f)
print("Telechargement WAV:")
for f in ["llm_opera_master.wav","llm_house_master.wav"]:
    if os.path.exists(f): print(" ",f); files.download(f)


In [ ]:
# Optionnel : MuseScore General SF3 (meilleure qualite, ~200MB)
# Decommenter pour utiliser

# !wget -q -O /tmp/MuseScore.sf3 \
#   "https://ftp.osuosl.org/pub/musescore/soundfont/MuseScore_General/MuseScore_General.sf3"
# render_midi(MIDI_OPERA,"llm_opera_hq.wav",sf="/tmp/MuseScore.sf3")
# master("llm_opera_hq.wav","llm_opera_hq_master.wav","orchestra")
# from google.colab import files; files.download("llm_opera_hq_master.wav")

print("Cellule optionnelle — decommenter pour MuseScore General SF3")
print("Qualite orchestrale superieure a la soundfont GM par defaut")
